In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import tempfile
from pathlib import Path
import sys

PROJECT_DIR = str(Path.cwd().parent) + "/"
sys.path.append(PROJECT_DIR)

from mpp_project.daily_pipeline import run_daily_pipeline
from mpp_project.oracle_dp import GAP_OFFSET  # 600
# Helpers partagés avec la suite pytest (tests/helpers.py)
from tests.helpers import exact_theoretical_wr, terminal_strict, build_pipeau_dataframe, build_ghost_peloton

print("🧪 LABO DE TEST : run_daily_pipeline (bout-en-bout vs théorie exacte)")

# ==========================================
# 1. MATCH "PIPEAU" RÉPÉTÉ (N matchs de poules identiques)
# ==========================================
N_MATCHS   = 4
true_proba = np.array([0.60, 0.25, 0.15], dtype=np.float64)
crowd      = np.array([0.70, 0.20, 0.10], dtype=np.float64)
gains      = np.array([20, 50, 90], dtype=np.int64)   # PAIRS → pas d'arrondi en grille coarse

df = build_pipeau_dataframe(N_MATCHS, true_proba, crowd, gains)
tmp_csv = Path(tempfile.gettempdir()) / "_mpp_test_pipeline.csv"
df.to_csv(tmp_csv, index=False)

# ==========================================
# 2. INJECTIONS DE TEST (dependency injection via run_daily_pipeline)
# ==========================================
# Peloton FANTÔME (masse sur delta=0) + gap_2 saturé (+400) → seul Bob compte,
# réduit à un suivi 1D directement comparable au solveur récursif exact.
p_empirique_ghost = build_ghost_peloton(N_MATCHS, max_gain=250)
GAP_PELOTON_SAFE  = 400

print(f"✅ Setup OK — {N_MATCHS} matchs identiques | CSV temporaire : {tmp_csv}")

In [ ]:
# ==========================================
# 3. EXÉCUTION DU PIPELINE (sans drift → 100% déterministe)
# ==========================================
# horizon_nuit=1 : on ne calcule QUE la Q-table du match du jour (match 0).
# Sa condition limite V_next encode déjà toute la profondeur (matchs 1..3 +
# terminal) via la rétro-propagation coarse → décision évaluée à profondeur N_MATCHS.
reco, wr, ev, q_table_jour = run_daily_pipeline(
    csv_path=tmp_csv,
    match_id_cible=1,
    mon_gap_1=0,
    mon_gap_2=GAP_PELOTON_SAFE,
    has_booster=1,
    use_drift=False,                       # déterministe : comparable à la théorie exacte
    horizon_nuit=1,
    nb_scenarios=1,
    p_empirique_override=p_empirique_ghost,  # peloton fantôme
    save_abaques=False                       # ne pas polluer data/
)

idx_g2 = GAP_PELOTON_SAFE + GAP_OFFSET      # 400 + 600 = 1000


def test_pipeline(nom, gap_initial):
    idx_g1 = gap_initial + GAP_OFFSET

    # --- DP (lecture directe de la Q-table du jour, max sur les 3 outcomes) ---
    wr_base  = np.max(q_table_jour[idx_g1, idx_g2, :, 0])
    wr_keep  = np.max(q_table_jour[idx_g1, idx_g2, :, 1])
    wr_use   = np.max(q_table_jour[idx_g1, idx_g2, :, 2])
    wr_boost = max(wr_keep, wr_use)

    # --- Théorie exacte (helper partagé, terminal strict comme solve_dp_coarse) ---
    th_base  = exact_theoretical_wr(gap_initial, N_MATCHS, true_proba, crowd, gains,
                                    has_booster=False, terminal=terminal_strict)
    th_boost = exact_theoretical_wr(gap_initial, N_MATCHS, true_proba, crowd, gains,
                                    has_booster=True, terminal=terminal_strict)

    ok = abs(wr_base - th_base) < 0.005 and abs(wr_boost - th_boost) < 0.005
    print(f"▶️ {nom}  (retard sur Bob : {-gap_initial} pts)")
    print(f"   [Base] Théorie {th_base*100:6.2f}% | DP {wr_base*100:6.2f}%")
    print(f"   [ x2 ] Théorie {th_boost*100:6.2f}% | DP {wr_boost*100:6.2f}%")
    print("   ✅ RÉUSSI" if ok else "   ❌ ÉCHEC")
    print("-" * 56)


print("\n" + "=" * 56)
print(f"📊 TEST run_daily_pipeline — horizon {N_MATCHS} matchs")
print("=" * 56)

# Gain max base = 4×90 = 360 pts ; avec booster = 3×90 + 180 = 450 pts
test_pipeline("Mission impossible (même x2)", gap_initial=-460)
test_pipeline("Le booster, seule issue",      gap_initial=-400)
test_pipeline("Droit a l'erreur elargi",      gap_initial=-300)
test_pipeline("Retard modere",                gap_initial=-150)
test_pipeline("Gestion de l'avance",          gap_initial=50)